# 🐈 CatVTON — Inference
Virtual try-on on Colab T4 (16GB). Upload person + garment → get result.

In [6]:
#@title 1. Setup
!git clone https://github.com/Zheng-Chong/CatVTON.git /content/CatVTON 2>/dev/null; echo "done"
%cd /content/CatVTON

!pip install -q diffusers fvcore omegaconf pycocotools av gradio "peft>=0.17.0"

import torch, diffusers
print(f"✅ diffusers {diffusers.__version__} | torch {torch.__version__}")
print(f"✅ GPU: {torch.cuda.get_device_name(0)} (sm_{torch.cuda.get_device_capability(0)[0]}{torch.cuda.get_device_capability(0)[1]})")

done
/content/CatVTON
✅ diffusers 0.38.0 | torch 2.11.0+cu128
✅ GPU: Tesla T4 (sm_75)


In [7]:
#@title 2. Load Model
import os, torch
os.chdir('/content/CatVTON')

from huggingface_hub import snapshot_download
from diffusers.image_processor import VaeImageProcessor
from model.cloth_masker import AutoMasker, vis_mask
from model.pipeline import CatVTONPipeline
from utils import init_weight_dtype, resize_and_crop, resize_and_padding

# T4 = sm_75 → must use fp16 (bf16 needs sm_80+ for xformers/efficient kernels)
cc = torch.cuda.get_device_capability(0)
PRECISION = "bf16" if cc[0] >= 8 else "fp16"
print(f"GPU compute capability: {cc[0]}.{cc[1]} → using {PRECISION}")

repo_path = snapshot_download(repo_id="zhengchong/CatVTON")

pipeline = CatVTONPipeline(
    base_ckpt="booksforcharlie/stable-diffusion-inpainting",
    attn_ckpt=repo_path, attn_ckpt_version="mix",
    weight_dtype=init_weight_dtype(PRECISION),
    use_tf32=True, device='cuda'
)

# Fix OOM on T4: disable the math SDPA backend (materializes 36GB attention matrix)
# Keep mem_efficient SDPA which processes attention in chunks
# Do NOT use xformers — it overwrites CatVTON's custom SkipAttnProcessor
torch.backends.cuda.enable_flash_sdp(False)
torch.backends.cuda.enable_math_sdp(False)
torch.backends.cuda.enable_mem_efficient_sdp(True)
print("✅ SDPA: math=off, flash=off, mem_efficient=on")

mask_processor = VaeImageProcessor(vae_scale_factor=8, do_normalize=False, do_binarize=True, do_convert_grayscale=True)
automasker = AutoMasker(
    densepose_ckpt=os.path.join(repo_path, "DensePose"),
    schp_ckpt=os.path.join(repo_path, "SCHP"),
    device='cuda'
)

print(f"✅ Ready | VRAM: {torch.cuda.memory_allocated()/1024**3:.1f} GB")

GPU compute capability: 7.5 → using fp16


Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


Loading weights:   0%|          | 0/396 [00:00<?, ?it/s]

An error occurred while trying to fetch booksforcharlie/stable-diffusion-inpainting: booksforcharlie/stable-diffusion-inpainting does not appear to have a file named diffusion_pytorch_model.safetensors.
Defaulting to unsafe serialization. Pass `allow_pickle=False` to raise an error instead.


✅ SDPA: math=off, flash=off, mem_efficient=on
✅ Ready | VRAM: 5.6 GB


In [18]:
#@title 3. Upload Images
from google.colab import files
from IPython.display import display, HTML

display(HTML("<b>Upload Person Image:</b>"))
person_file = list(files.upload().keys())[0]

display(HTML("<b>Upload Garment Image:</b>"))
garment_file = list(files.upload().keys())[0]

Saving result.png to result (1).png


Saving shopping.webp to shopping.webp


In [ ]:
#@title 4. Run Try-On
#@markdown ---
cloth_type = "upper"  #@param ["upper", "lower", "overall"]
steps = 50  #@param {type:"slider", min:10, max:100, step:5}
guidance = 2.0  #@param {type:"slider", min:0, max:7.5, step:0.5}
seed = 42  #@param {type:"integer"}
#@markdown Set seed to -1 for random.
#@markdown ---

import time, gc, matplotlib.pyplot as plt
from PIL import Image

W, H = 768, 1024

gc.collect()
torch.cuda.empty_cache()

person_img = resize_and_crop(Image.open(person_file).convert("RGB"), (W, H))
cloth_img = resize_and_padding(Image.open(garment_file).convert("RGB"), (W, H))

mask = automasker(person_img, cloth_type)['mask']
mask = mask_processor.blur(mask, blur_factor=9)

gen = torch.Generator(device='cuda').manual_seed(seed) if seed != -1 else None

t0 = time.time()
result = pipeline(
    image=person_img, condition_image=cloth_img, mask=mask,
    num_inference_steps=steps, guidance_scale=guidance, generator=gen
)[0]
print(f"Done in {time.time()-t0:.1f}s | Peak VRAM: {torch.cuda.max_memory_allocated()/1024**3:.1f} GB")

fig, ax = plt.subplots(1, 4, figsize=(20, 7))
for a, img, t in zip(ax, [person_img, vis_mask(person_img, mask), cloth_img, result],
                      ['Person', 'Mask', 'Garment', 'Result']):
    a.imshow(img); a.set_title(t, fontsize=14, fontweight='bold'); a.axis('off')
plt.tight_layout(); plt.show()

result.save('result.png')
print("Saved: result.png")

  6%|▌         | 3/50 [00:07<02:04,  2.64s/it]

In [20]:
#@title 5. Download Result
files.download('result.png')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>